# Multi-class Classification of Car Origin

**Objective:** Predict car origin classes with a multi-class logistic regression workflow.

**Dataset used:** `data/auto.csv`

**Main techniques used:** Categorical encoding, train-test split, one-vs-rest style classification, probability comparison


In [3]:
import pandas as pd
cars = pd.read_csv('data/auto.csv')
print(cars.head())


    mpg  cylinders  displacement  horsepower  weight  acceleration  year  \
0  18.0          8         307.0       130.0  3504.0          12.0    70   
1  15.0          8         350.0       165.0  3693.0          11.5    70   
2  18.0          8         318.0       150.0  3436.0          11.0    70   
3  16.0          8         304.0       150.0  3433.0          12.0    70   
4  17.0          8         302.0       140.0  3449.0          10.5    70   

   origin  
0       1  
1       1  
2       1  
3       1  
4       1  


In [4]:
unique_regions = cars ['origin'].unique()
print(unique_regions)


[1 3 2]


In [5]:
cars['cylinders'].value_counts()


cylinders
4    199
8    103
6     83
3      4
5      3
Name: count, dtype: int64

In [6]:
cars['year'].value_counts()


year
73    40
78    36
76    34
82    30
75    30
79    29
70    29
72    28
77    28
81    28
71    27
80    27
74    26
Name: count, dtype: int64

In [7]:
#panda.get_dummies()
dummy_df = pd.get_dummies(cars['cylinders'], prefix='cyl')
cars = pd.concat([cars, dummy_df], axis=1)
print(cars.head())


    mpg  cylinders  displacement  horsepower  weight  acceleration  year  \
0  18.0          8         307.0       130.0  3504.0          12.0    70   
1  15.0          8         350.0       165.0  3693.0          11.5    70   
2  18.0          8         318.0       150.0  3436.0          11.0    70   
3  16.0          8         304.0       150.0  3433.0          12.0    70   
4  17.0          8         302.0       140.0  3449.0          10.5    70   

   origin  cyl_3  cyl_4  cyl_5  cyl_6  cyl_8  
0       1  False  False  False  False   True  
1       1  False  False  False  False   True  
2       1  False  False  False  False   True  
3       1  False  False  False  False   True  
4       1  False  False  False  False   True  


In [ ]:
dummy_years = pd.get_dummies(cars['year'], prefix='year')
cars = pd.concat([cars, dummy_years], axis=1)
print(cars.head())
cars.drop(['cylinders', 'year'], axis=1, inplace=True)


In [12]:
print(cars.head())


    mpg  displacement  horsepower  weight  acceleration  origin  cyl_3  cyl_4  \
0  18.0         307.0       130.0  3504.0          12.0       1  False  False   
1  15.0         350.0       165.0  3693.0          11.5       1  False  False   
2  18.0         318.0       150.0  3436.0          11.0       1  False  False   
3  16.0         304.0       150.0  3433.0          12.0       1  False  False   
4  17.0         302.0       140.0  3449.0          10.5       1  False  False   

   cyl_5  cyl_6  ...  year_73  year_74  year_75  year_76  year_77  year_78  \
0  False  False  ...    False    False    False    False    False    False   
1  False  False  ...    False    False    False    False    False    False   
2  False  False  ...    False    False    False    False    False    False   
3  False  False  ...    False    False    False    False    False    False   
4  False  False  ...    False    False    False    False    False    False   

   year_79  year_80  year_81  year_82  
0   

In [ ]:
import numpy as np 
shuffled_rows = np.random.permutation(cars.index)
shuffled_cars = cars.iloc[shuffled_rows]

highest_train_row = int(cars.shape[0] *.7)
train = shuffled_cars.iloc[0:highest_train_row]
test = shuffled_cars.iloc[highest_train_row:]
print(train.shape)
print(test.shape)
#on a maintenant 70% des données pour l'entrainement et 30% pour le test


(274, 37)
(118, 37)


In [15]:
#str.startswith pour selectionner les colonnes qui commencent par 'cyl' ou 'year'
from sklearn.linear_model import LogisticRegression
unique_regions = cars ['origin'].unique()
unique_regions.sort()
models = {}
features = [col for col in train.columns if col.startswith('cyl') or col.startswith('year')]
for region in unique_regions:
    model = LogisticRegression(solver='liblinear')
    x_train = train[features]
    y_train = (train['origin'] == region)
    model.fit(x_train, y_train)
    models[region] = model
#on a entrainé un modèle de régression logistique pour chaque région d'origine des voitures
#pour faire des prédictions, on peut utiliser les modèles entrainés pour chaque région


In [16]:
#on va tester les modèles sur les données de test
testing_probs = pd.DataFrame(columns=unique_regions)
for origin in unique_regions:
    #selectioner les caractéristiques de test
    x_test = test[features]
    #calc la proba d'observations de la variable iteratrice origin
    testing_probs[origin] = models[origin].predict_proba(x_test)[:,1]
testing_probs.head()


,1,2,3
0,0.901059,0.047366,0.051852
1,0.344788,0.373866,0.271475
2,0.923643,0.005259,0.178991
3,0.269153,0.487556,0.247328
4,0.928637,0.030260,0.058395


In [17]:
#on va choisir l'origine de la voituez en fonction de la proba la plus élevée
# utiliser .idmax(axis=1) pour trouver la colonne avec la proba la plus élevée pour chaque ligne
predicted_origins = testing_probs.idxmax(axis=1)
print(predicted_origins[0:5])
testing_probs.head()


0    1
1    2
2    1
3    2
4    1
dtype: int64


,1,2,3
0,0.901059,0.047366,0.051852
1,0.344788,0.373866,0.271475
2,0.923643,0.005259,0.178991
3,0.269153,0.487556,0.247328
4,0.928637,0.030260,0.058395


## Conclusion

**What was done:** Encoded categorical features and predicted the origin of cars with a multi-class classification workflow.

**What was learned:** Proper feature encoding is essential when applying linear classifiers to multi-class problems.

**Possible improvements:** Evaluate the model with a confusion matrix and compare it with tree-based or ensemble classifiers.
